# One-off: move the tables into the bronze and silver schemas

Job `migrate_to_layer_schemas`. Run it **once per target, before the first run of `ingest_financebench` in the new layout**, and delete the job afterwards.

Before, one schema held everything. Now bronze and silver have a schema each. The PDFs and `bronze_pdfs` cost nothing to rebuild (the download is free and Auto Loader reloads them), but the silver tables hold parser and model calls that were paid for. This job copies them, so nothing is parsed or converted twice:

| From (old schema) | To |
| --- | --- |
| `silver_parts` | `bronze.bronze_pdf_parts` |
| `silver_parsed` | `silver.silver_parsed` |
| `silver_pages` | `silver.silver_pages` |
| `silver_log` | `silver.silver_log` |

**The paths change.** The `path` column holds the full volume path, which includes the schema (`/Volumes/<catalog>/<old schema>/docs/pdfs/X.pdf`). The PDFs are downloaded again into the new bronze volume, so every copied `path` is rewritten to the new volume. Without this, no key would match and everything would be parsed again.

Each table is copied only if the new one is empty, so running the job twice changes nothing. The old schema is not touched: drop it yourself, after the checks at the end show the same counts.

In [ ]:
import sys

sys.path.insert(0, "../common")
sys.path.insert(0, "../bronze")
sys.path.insert(0, "../silver")
from bronze_tables import create_parts_table
from names import Names
from silver_tables import create_silver_tables

dbutils.widgets.text("catalog", "workspace")
dbutils.widgets.text("old_schema", "finance_bench_rag")
dbutils.widgets.text("bronze_schema", "finance_bench_bronze")
dbutils.widgets.text("silver_schema", "finance_bench_silver")

catalog = dbutils.widgets.get("catalog")
old = f"{catalog}.{dbutils.widgets.get('old_schema')}"
n = Names(catalog, dbutils.widgets.get("bronze_schema"), dbutils.widgets.get("silver_schema"))
old_volume = f"/Volumes/{catalog}/{dbutils.widgets.get('old_schema')}/"
new_volume = f"/Volumes/{catalog}/{n.bronze_schema}/"

spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.{n.bronze_schema}")
spark.sql(f"CREATE VOLUME IF NOT EXISTS {catalog}.{n.bronze_schema}.docs")
create_parts_table(spark, n)
create_silver_tables(spark, n)

In [ ]:
# (old table, new table, [(column of the new table, how it is read from the old one)])
NEW_PATH = f"replace(path, '{old_volume}', '{new_volume}')"
copies = [
    ("silver_parts", n.parts, [("path", NEW_PATH), "part_no", "page_offset", "pages", "content", "split_at"]),
    ("silver_parsed", n.parsed, [("path", NEW_PATH), ("part_no", "coalesce(part_no, 1)"), ("page_offset", "coalesce(page_offset, 0)"),
                                 "parsed", "pdf_pages", "pages", "parsed_at"]),
    ("silver_pages", n.pages, [("path", NEW_PATH), "page_no", "markdown", "converted_at"]),
    ("silver_log", n.log, [("path", NEW_PATH), "stage", "page_no", "attempt", "retryable", "reason", "pdf_pages", "logged_at", "part_no"]),
]
copied = []  # the tables copied in this run
for name, target, columns in copies:
    columns = [(c, c) if isinstance(c, str) else c for c in columns]
    if not spark.catalog.tableExists(f"{old}.{name}"):
        print(f"{name}: not in the old schema, skipped")
    elif spark.table(target).limit(1).count() > 0:
        print(f"{target}: not empty, skipped")
    else:
        spark.sql(f"""
        INSERT INTO {target} ({", ".join(c for c, _ in columns)})
        SELECT {", ".join(e for _, e in columns)} FROM {old}.{name}""")
        copied.append(name)
        print(f"{name} -> {target}")

In [ ]:
# The check: the counts must be the same on both sides. Only for the tables copied in this run: on a rerun
# the new tables have grown (the pipeline kept running in them).
for name, target, _ in copies:
    if name in copied:
        before, after = spark.table(f"{old}.{name}").count(), spark.table(target).count()
        print(f"{'OK  ' if before == after else 'FAIL'} {name}: {before} rows before, {after} after")
        assert before == after, f"{name}: the copy is not complete"
stale = spark.sql(f"SELECT count(*) FROM {n.parsed} WHERE path NOT LIKE '%{new_volume}%'").first()[0]
print(f"{'OK  ' if stale == 0 else 'FAIL'} silver_parsed paths that do not start with the new volume: {stale}")
assert stale == 0

## Fill the bytes of the parts that were parsed before the split

A PDF parsed before the split was copied as one whole part **without bytes** (nothing reads them: the part is parsed). A rebuild of silver would need them, so once `bronze_pdfs` is loaded in the new volume, the whole PDF's bytes are put into those parts.

In [ ]:
if spark.catalog.tableExists(n.bronze):
    spark.sql(f"""
    MERGE INTO {n.parts} t USING (SELECT path, content FROM {n.bronze}) b
    ON t.path = b.path AND t.part_no = 1 AND t.content IS NULL
    WHEN MATCHED THEN UPDATE SET t.content = b.content""")
    without_bytes = spark.sql(f"SELECT count(*) FROM {n.parts} WHERE content IS NULL").first()[0]
    print(f"{'OK  ' if without_bytes == 0 else 'FAIL'} parts without bytes: {without_bytes}")
    assert without_bytes == 0
else:
    print("bronze_pdfs is not loaded yet: run ingest_financebench first, then this job again")